In [22]:
import pandas as pd
from genai.client import Client
from genai.credentials import Credentials
from genai.schema import (
    TextGenerationParameters,
    TextGenerationReturnOptions,
    DecodingMethod
)
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
import ray
import json
import numpy as np
import random

In [6]:
load_dotenv()

True

In [7]:
ray.init(num_cpus=8)

2024-04-04 10:25:54,052	INFO worker.py:1715 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.13
Ray version:,2.9.0
Dashboard:,http://127.0.0.1:8265


In [8]:
train_df = pd.read_csv('../fmea_result/autoQ_train_data_input_for_experiments.csv')
val_df = pd.read_csv('../fmea_result/autoQ_val_data_input_for_experiments.csv')
test_df = pd.read_csv('../fmea_result/autoQ_test_data_input_for_experiments.csv')

In [21]:
print(train_df.iloc[0].long_description)

The equipment Accumulator - Pneumatic - Bladder Type, is categorized as Fixed Asset and has the following boundary: A Pneumatic Accumulator - Bladder Type in this database is comprised of:  - Tank - Bladder - Air Line Check Valve, if present - Gas Precharge Valve


In [5]:
train_df = pd.read_csv('train_llama.csv')
val_df = pd.read_csv('val_llama.csv')
test_df = pd.read_csv('test_llama.csv')

In [6]:
num_shuffle = 3
num_missing = 3

In [20]:
def shuffle_components(df):
    augmented_rows = []
    for i, row in df.iterrows():
        failure_locations = list(eval(row['failure_locations']))
        for _ in range(num_shuffle):
            row = row.copy()
            shuffled_locations = failure_locations.copy()
            random.shuffle(shuffled_locations)
            row.failure_locations = shuffled_locations
            augmented_rows.append(row)
    return df

def drop_components(df):
    augmented_rows = []
    for i, row in df.iterrows():
        failure_locations = list(eval(row['failure_locations']))
        for _ in range(num_missing):
            missing_items = random.choice(failure_locations)
            remaining_items = failure_locations.copy()
            remaining_items.remove(missing_items)
            row.missing_items = missing_items
            row.failure_locations = remaining_items
            augmented_rows.append(row)
    df = pd.DataFrame(augmented_rows).reset_index().drop('index', axis=1)
    return df

def drop_boundary_locations(df):
    df = df.copy()
    keywords = ['following:', 'of:', 'includes:', 'includes the', 'include:', 'only the', 'is:', 
                'comprises:', 'comprised:', 'comprising:', 'The boundary of the', 'boundary:']
    mask = [False] * len(df)
    sliced_descriptions = []
    list_idxs = []
    for i, row in df.iterrows():
        long_desc = row.long_description
        for keyword in keywords:
            idx = long_desc.find(keyword)
            if idx != -1:
                idx += len(keyword)
                break
        list_idxs.append(idx)
        sliced_descriptions.append(long_desc[idx:])
    df['long_description_sliced'] = sliced_descriptions
    df['boundary_loc_idx'] = list_idxs
    return df

def generate_failure_modes(df):
    pass

def augment_df(df):
    return pd.concat([shuffle_components(df), drop_components(df)])

In [16]:
for i in range(len(train_df)):
    print(train_df.long_description_sliced.iloc[i])

  - Tank - Bladder - Air Line Check Valve, if present - Gas Precharge Valve
  -Ejector and internals from the suction to discharge ports -Steam inlet port and nozzle, including steam chest, if present -Ejector steam condensers, if present  All valves are excluded from the boundary.
  Ammonia vaporizer Ammonia vaporizer heater Piping Valves: relief and block Level indicators and transmitters Pressure sensors and transmitters Thermocouples Dilution air blower and motor (a more detailed treatment of relevant motors may be found elsewhere in the database under "Motor - Low Voltage"  Excludes:  Ammonia Transfer Pump and motor Flow Control Valve - AOV actuators can be found elsewhere in the database under "Valve - Air Operated - AOV" Flange connections Supply system boundaries (electrical, steam, glycol, etc.) Controls Wiring to control system
 - Autoclave (Chamber, Door, Closing Device, Seal or Gasket, Steam Jacket, Insulation) - Control Panel and Enclosure (Control Module, Disconnect, Main

In [239]:
print(train_df.long_description.iloc[0])

The equipment Accumulator - Pneumatic - Bladder Type, is categorized as Fixed Asset and has the following boundary: A Pneumatic Accumulator - Bladder Type in this database is comprised of:  - Tank - Bladder - Air Line Check Valve, if present - Gas Precharge Valve


In [93]:
train_df = augment_df(train_df)
val_df = augment_df(val_df)
test_df = augment_df(test_df)

In [82]:
idx=1
print(train_df.long_description.iloc[idx])
print(train_df.failure_locations.iloc[idx])

The equipment Steam Jet Air Ejector, is categorized as Fixed Asset and has the following boundary: The boundary of a typical steam jet air ejector boundary for the purpose of this database consists of:  -Ejector and internals from the suction to discharge ports -Steam inlet port and nozzle, including steam chest, if present -Ejector steam condensers, if present  All valves are excluded from the boundary.
{'Condenser - Internal Hardware; including: Baffle Plates, Support Plates, Tie Rods, Spacers, Diffusers Plates, and Impingement Plates', 'Condenser - Tube Sheets', 'Condenser - Tube Joint: Rolled or welded', 'Condenser - Closure Devices Channel Partitions, Manways, and Flange', 'Ejector - Steam Nozzle', 'Condenser - Closure Devices Channel Partitions, Manways,  and Flange', 'Condenser - Tube Joint: Rolled', 'Condenser - Shell, Inlet and Outlet Nozzles', 'Ejector - Throat and Diffuser', 'Condenser - Tubes'}


In [6]:
type_col = 'TypeData.GenCompType'
long_desc_col = 'long_description'
failure_loc_col = 'failure_locations'

In [7]:
template = f"""
I have an industrial asset, its long description and the places where the asset can fails. Your task is to generate a readable summary of failure locations only using provided information in failure location. 

Asset: <asset>

Long Description: <long_desc>

Failure Locations: <failure_loc>

Here is a readable Summary of Failure Locations:
"""

In [8]:
def get_prompts(df):
    prompts = []
    for idx, row in df.iterrows():
        asset = row[type_col]
        long_desc = row[long_desc_col]
        failure_loc = str(eval(row[failure_loc_col])).replace('{', '').replace('}', '').replace("'", "")
        prompt = template.replace('<asset>', asset) \
                         .replace('<long_desc>', long_desc) \
                         .replace('<failure_loc>', failure_loc)
        prompts.append(prompt)
    return prompts

In [9]:
limit = None

In [10]:
train_prompts = get_prompts(df_train.iloc[:limit])
val_prompts = get_prompts(df_val[:limit])
test_prompts = get_prompts(df_test[:limit])

In [11]:
# summaries = []
# for prompt in train_prompts:
#     model_id = "mistralai/Mixtral-8x7B-v0.1"
#     tokenizer = AutoTokenizer.from_pretrained(model_id)
#     model = AutoModelForCausalLM.from_pretrained(model_id, use_flash_attention_2=True)
#     inputs = tokenizer(prompt, return_tensors="pt").to(0)
#     outputs = model.generate(**inputs, max_new_tokens=20)
#     summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print(summary)
#     summaries.append(summary)


In [12]:
@ray.remote
def get_summaries(prompt):
    client = Client(credentials=Credentials.from_env())
    results = client.text.generation.create(
        model_id="mistralai/mixtral-8x7b-instruct-v0-1",
        inputs=[prompt],
        parameters=TextGenerationParameters(
            decoding_method=DecodingMethod.GREEDY,
            return_options=TextGenerationReturnOptions(
                input_text=True,
            ),
        ),
    )
    return next(iter(results)).results[0].generated_text

In [ ]:
remote_call = []
for pt in train_prompts:
    remote_call.append(get_summaries.remote(pt))
train_summaries = ray.get(remote_call)
print('train generated')

(get_summaries pid=73509) Exception raised during processing
(get_summaries pid=73509) Failed to handle request after 3 retries to https://bam-api.res.ibm.com/v2/text/generation?version=2024-03-19.
(get_summaries pid=73509) {
(get_summaries pid=73509)   "error": "Too Many Requests",
(get_summaries pid=73509)   "extensions": {
(get_summaries pid=73509)     "code": "TOO_MANY_REQUESTS",
(get_summaries pid=73509)     "state": null,
(get_summaries pid=73509)     "reason": "CONCURRENCY_LIMIT"
(get_summaries pid=73509)   },
(get_summaries pid=73509)   "message": "Maximum of your concurrent requests exceeded",
(get_summaries pid=73509)   "status_code": 429
(get_summaries pid=73509) }
(get_summaries pid=73509) You can't send more requests due to the API concurrency limit. There is another running process (probably). Retrying in 100 ms.
(get_summaries pid=73509) You can't send more requests due to the API concurrency limit. There is another running process (probably). Retrying in 100 ms.
(get_su

RayTaskError(ApiResponseException): [36mray::get_summaries()[39m (pid=73509, ip=127.0.0.1)
  File "/Users/christodoulosconstantinides/miniconda3/lib/python3.10/site-packages/httpx/_models.py", line 759, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '429 Too Many Requests' for url 'https://bam-api.res.ibm.com/v2/text/generation?version=2024-03-19'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

The above exception was the direct cause of the following exception:

[36mray::get_summaries()[39m (pid=73509, ip=127.0.0.1)
  File "/var/folders/1z/rhghrxrd7ds2qf_kf_c6z7yw0000gn/T/ipykernel_73487/3259783961.py", line 14, in get_summaries
  File "/Users/christodoulosconstantinides/miniconda3/lib/python3.10/site-packages/genai/text/generation/generation_service.py", line 226, in create
    yield from execute_async(
  File "/Users/christodoulosconstantinides/miniconda3/lib/pyt

(get_summaries pid=73507) You can't send more requests due to the API concurrency limit. There is another running process (probably). Retrying in 100 ms.
(get_summaries pid=73507) Exception raised during processing
(get_summaries pid=73507) Failed to handle request after 3 retries to https://bam-api.res.ibm.com/v2/text/generation?version=2024-03-19.
(get_summaries pid=73507) {
(get_summaries pid=73507)   "error": "Too Many Requests",
(get_summaries pid=73507)   "extensions": {
(get_summaries pid=73507)     "code": "TOO_MANY_REQUESTS",
(get_summaries pid=73507)     "state": null,
(get_summaries pid=73507)     "reason": "CONCURRENCY_LIMIT"
(get_summaries pid=73507)   },
(get_summaries pid=73507)   "message": "Maximum of your concurrent requests exceeded",
(get_summaries pid=73507)   "status_code": 429
(get_summaries pid=73507) }
2024-04-02 18:55:03,284	ERROR worker.py:405 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::get_summaries() (pid=73505, ip=127.0.0.1)
  

In [ ]:
with open('train_summaries.json', 'w') as f:
    json.dump({'results':train_summaries}, f)

In [ ]:
remote_call = []
for pt in val_prompts:
    remote_call.append(get_summaries.remote(pt))
val_summaries = ray.get(remote_call)
print('val generated')

In [ ]:
with open('val_summaries.json', 'w') as f:
    json.dump({'results':val_summaries}, f)

In [ ]:
remote_call = []
for pt in test_prompts:
    remote_call.append(get_summaries.remote(pt))
test_summaries = ray.get(remote_call)
print('test generated')

In [ ]:
with open('test_summaries.json', 'w') as f:
    json.dump({'results':test_summaries}, f)